In [2]:
from sqlalchemy import create_engine

In [3]:
# import 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

products = pd.read_csv('C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/products.csv')
products.head()

products.info()

# remove dupplicate values
products = products.drop_duplicates(subset='product_id')

# clean the product name
products['product_name'] = products['product_name'].str.title().str.strip()

# clean category
products['category'] = products['category'].str.strip()

# clean sub_category   
products['sub_category'] = products['sub_category'].str.strip()

# clean brand
products = products.dropna(subset='brand')

# clean outliers 
# business logic 
products = products[(products['cost_price'] > 0)]

# outlier removal
min_cost = products['cost_price'].quantile(0.25)
max_cost = products['cost_price'].quantile(0.75)

cost_iqr = max_cost - min_cost

lower_cost = min_cost - 1.5 * cost_iqr
upper_cost = max_cost + 1.5 * cost_iqr
# used to keep the outliers if the business needed in future 
outlier_cost = products[
    (products['cost_price'] < lower_cost) |
    (products['cost_price'] > upper_cost) 
]

products = products[
    (products['cost_price'] > lower_cost) &
    (products['cost_price'] < upper_cost) 
]


products['cost_price'].value_counts().sort_values()

products = products[products['cost_price'] > 0]

products = products[products['selling_price'] > 0]


products['selling_price'].min()


products['selling_price'].min()

# profit or not 
products['profit'] = products['selling_price'] - products['cost_price']

products['profit'] = products['profit'].round(2)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2030 entries, 0 to 2029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     2030 non-null   object 
 1   product_name   2030 non-null   object 
 2   category       2030 non-null   object 
 3   sub_category   2030 non-null   object 
 4   brand          1987 non-null   object 
 5   vendor_id      2030 non-null   object 
 6   cost_price     2030 non-null   float64
 7   selling_price  2030 non-null   float64
dtypes: float64(2), object(6)
memory usage: 127.0+ KB


In [4]:
conn = sqlite3.connect('managing.db')

In [5]:
# to sql add
products.to_sql('products', conn, if_exists='replace', index=False)
pd.read_sql_query('select * from products limit 2', conn)

,product_id,product_name,category,sub_category,brand,vendor_id,cost_price,selling_price,profit
0,PROD01357,Pro Dell Perfume,Beauty,Perfume,Dell,VEND0080,16695.05,25525.36,8830.31
1,PROD00985,Essential Samsung Bags,Fashion,Bags,Samsung,VEND0171,43026.11,49153.85,6127.74


In [6]:
# Sql
# Top 10 most expensive products.
# Average selling price by category.
# Products with invalid prices.
# Products whose selling price is below cost.
# Rank products within each category by price.
# Find categories with the highest average margin.

# Top 10 most expensive products.
pd.read_sql_query('with rnk_data as (select product_id, product_name, cost_price, dense_rank() over(order by cost_price desc) as rnk from products) select * from rnk_data where rnk <= 10 ',conn)

# Average selling price by category.
pd.read_sql_query('select category, avg(selling_price) as avg_price from products group by category order by avg_price desc' , conn)

# Products with invalid prices.
pd.read_sql_query('select product_id, cost_price from products where cost_price < 0 or cost_price is null', conn)

# Products whose selling price is below cost.
pd.read_sql_query("select product_id , selling_price, cost_price from products where selling_price < cost_price", conn)

# Rank products within each category by price.
pd.read_sql_query('select category, product_name, selling_price , dense_rank() over(partition by category order by selling_price desc) as price_rank from products', conn)

# Find categories with the highest average margin.
pd.read_sql_query("""
select category,
avg(profit) as avg_margin
from products 
where profit > 0
group by category
order by avg_margin desc   
""", conn)


,category,avg_margin
0,Sports,18615.275129
1,Beauty,18108.980040
2,Electronics,16618.029156
3,Home,13577.288636
4,Fashion,13374.603878
5,Books,9431.314436


In [7]:
# mysql settings
USER = "root"
PASSWORD = 'Abhi8383055393'
HOST = 'localhost'
PORT = '3306'
DATABASE = "ecommmerce_analysis"

In [8]:
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

In [9]:
mysql_engine = create_engine(connection_string)

In [10]:
products.to_sql('products', mysql_engine,if_exists='replace',index=False)

1758

In [11]:
products.to_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/products.csv", index=False)